# EvalHub API Demo — Model Evaluation on RHOAI

This notebook demonstrates how to use the **EvalHub REST API** to evaluate LLMs served via MaaS (Models-as-a-Service) against standardized benchmark suites.

**What you'll do:**
1. Connect to EvalHub and MaaS APIs
2. Browse available benchmark suites and models
3. Select benchmarks and submit evaluation jobs
4. Monitor progress and view results

**Prerequisites:**
- A Kubernetes secret named `eval-hub-api-key` with field `api-key` containing your MaaS API key
- Access to the RHOAI dashboard (Evaluations page shows results)

## Cell 1 — Setup & Configuration

In [ ]:
import os, json, time, warnings, subprocess
import requests
from IPython.display import display, HTML, Markdown, clear_output

warnings.filterwarnings("ignore", "Unverified HTTPS request")

# ── Cluster config ──────────────────────────────────────────
RHOAI_DASHBOARD = "https://rh-ai.apps.ocp.cloud.rhai-tmm.dev"
MAAS_BASE       = "https://maas.apps.ocp.cloud.rhai-tmm.dev"
EVALHUB_API     = f"{RHOAI_DASHBOARD}/eval-hub/api/v1"
NAMESPACE       = os.popen("cat /var/run/secrets/kubernetes.io/serviceaccount/namespace 2>/dev/null || echo user-micyang").read().strip()
SECRET_NAME     = "eval-hub-api-key"   # K8s secret with field 'api-key'

# ── Auth (use SA token inside workbench) ───────────────────
SA_TOKEN_PATH = "/var/run/secrets/kubernetes.io/serviceaccount/token"
if os.path.exists(SA_TOKEN_PATH):
    with open(SA_TOKEN_PATH) as f:
        OCP_TOKEN = f.read().strip()
else:
    OCP_TOKEN = subprocess.check_output(["oc", "whoami", "-t"]).decode().strip()

HEADERS = {
    "Authorization": f"Bearer {OCP_TOKEN}",
    "Content-Type": "application/json",
}

# ── Verify connectivity ────────────────────────────────────
r = requests.get(f"{EVALHUB_API}/evalhub/health", params={"namespace": NAMESPACE}, headers=HEADERS, verify=False)
health = r.json().get("data", {})
print(f"Namespace:  {NAMESPACE}")
print(f"EvalHub:    {health.get('status', 'unknown')} (available={health.get('available', False)})")

# Verify MaaS
MAAS_KEY = os.environ.get("OPENAI_API_KEY", "")
if not MAAS_KEY:
    try:
        MAAS_KEY = subprocess.check_output(
            ["oc", "get", "secret", SECRET_NAME, "-n", NAMESPACE,
             "-o", "jsonpath={.data.api-key}"],
        ).decode().strip()
        import base64; MAAS_KEY = base64.b64decode(MAAS_KEY).decode()
    except Exception:
        MAAS_KEY = ""

if MAAS_KEY:
    mr = requests.get(f"{MAAS_BASE}/v1/models", headers={"Authorization": f"Bearer {MAAS_KEY}"}, verify=False)
    models = [m["id"] for m in mr.json().get("data", []) if m.get("ready")]
    print(f"MaaS:       {len(models)} models available")
else:
    models = []
    print("MaaS:       key not found (models will be entered manually)")

print(f"\n✅ Ready!")

## Cell 2 — EvalHub Helper Functions

In [ ]:
def evalhub_get(path, **params):
    """GET request to EvalHub API."""
    params.setdefault("namespace", NAMESPACE)
    r = requests.get(f"{EVALHUB_API}/{path}", params=params, headers=HEADERS, verify=False)
    r.raise_for_status()
    return r.json()

def evalhub_post(path, payload, **params):
    """POST request to EvalHub API."""
    params.setdefault("namespace", NAMESPACE)
    r = requests.post(f"{EVALHUB_API}/{path}", json=payload, params=params, headers=HEADERS, verify=False)
    r.raise_for_status()
    return r.json()

def evalhub_delete(path, **params):
    """DELETE request to EvalHub API."""
    params.setdefault("namespace", NAMESPACE)
    r = requests.delete(f"{EVALHUB_API}/{path}", params=params, headers=HEADERS, verify=False)
    r.raise_for_status()
    return r.json()

def list_collections():
    """Return available benchmark suites."""
    data = evalhub_get("evaluations/collections", limit=50)
    return data["data"]["items"]

def list_jobs():
    """Return current evaluation jobs."""
    data = evalhub_get("evaluations/jobs")
    return data.get("data", [])

def create_job(name, model_name, model_url, secret_ref, benchmarks):
    """Submit an evaluation job.
    
    Args:
        name: Display name for the job
        model_name: Model ID (e.g. 'publishers/prelude-maas/models/gemma4')
        model_url: Base URL (e.g. 'https://maas.../v1')
        secret_ref: K8s secret name containing 'api-key'
        benchmarks: List of dicts with 'id' and 'provider_id'
    """
    payload = {
        "name": name,
        "model": {
            "name": model_name,
            "url": model_url,
            "auth": {"secret_ref": secret_ref},
        },
        "benchmarks": benchmarks,
    }
    return evalhub_post("evaluations/jobs", payload)

def wait_for_job(job_id, poll_interval=15, timeout=600):
    """Poll until a job completes or fails."""
    start = time.time()
    while time.time() - start < timeout:
        jobs = list_jobs()
        job = next((j for j in jobs if j["resource"]["id"] == job_id), None)
        if job is None:
            print(f"⚠ Job {job_id} not found in list")
            return None
        state = job["status"]["state"]
        elapsed = int(time.time() - start)
        clear_output(wait=True)
        print(f"⏳ {job['name']}  state={state}  ({elapsed}s)")
        if state in ("complete", "completed", "failed", "error"):
            return job
        time.sleep(poll_interval)
    print("⏰ Timeout")
    return None

print("✅ Helper functions loaded")

## Cell 3 — Browse Benchmark Suites

EvalHub provides pre-built **benchmark suites** (collections). Each suite bundles multiple benchmarks with scoring weights and pass criteria.

In [ ]:
collections = list_collections()

for i, c in enumerate(collections):
    r = c["resource"]
    benchmarks = c.get("benchmarks", [])
    threshold = c.get("pass_criteria", {}).get("threshold", "N/A")
    
    print(f"{'='*60}")
    print(f"  [{i}] {c['name']}")
    print(f"      ID:        {r['id']}")
    print(f"      Category:  {c.get('category', 'N/A')}")
    print(f"      Threshold: {threshold}")
    print(f"      Tags:      {', '.join(c.get('tags', []))}")
    print(f"      Benchmarks:")
    for b in benchmarks:
        metric = b.get("primary_score", {}).get("metric", "?")
        b_thresh = b.get("pass_criteria", {}).get("threshold", "?")
        print(f"        • {b['id']:45s}  metric={metric}, pass≥{b_thresh}")
    print()

# Store for later use
COLLECTIONS = {c["resource"]["id"]: c for c in collections}
print(f"\n📋 {len(collections)} benchmark suites available")

## Cell 4 — Browse Available Models

Models served via MaaS that can be evaluated.

In [ ]:
if MAAS_KEY:
    mr = requests.get(f"{MAAS_BASE}/v1/models",
                      headers={"Authorization": f"Bearer {MAAS_KEY}"},
                      verify=False)
    all_models = mr.json().get("data", [])
else:
    all_models = []

MODELS = []
for i, m in enumerate(all_models):
    if m.get("ready") and m.get("kind") == "LLMInferenceService":
        MODELS.append(m["id"])
        short = m["id"].split("/")[-1]
        print(f"  [{i}] {short:30s}  ({m['id']})")

print(f"\n🤖 {len(MODELS)} models available for evaluation")

## Cell 5 — Configure Your Evaluation

Pick a model and one or more benchmarks. You can run a **full suite** or **individual benchmarks**.

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CONFIGURE YOUR EVALUATION HERE                         ║
# ╚══════════════════════════════════════════════════════════╝

# ── Model ──────────────────────────────────────────────────
EVAL_MODEL = "publishers/prelude-maas/models/gemma4"

# ── Benchmarks (pick individual benchmarks) ───────────────
#    Option A: Use a full suite
#    EVAL_SUITE = "leaderboard-v2"          # full Open LLM Leaderboard
#    EVAL_SUITE = "safety-and-fairness-v1"  # full Safety & Fairness
#    EVAL_SUITE = "toxicity-and-ethical-principles"
#
#    Option B: Pick individual benchmarks from any suite
EVAL_SUITE = None  # set to None when using custom benchmarks below

EVAL_BENCHMARKS = [
    # -- Open LLM Leaderboard v2 --
    "leaderboard_ifeval",         # instruction following (generate_until)
    # "leaderboard_bbh",          # BIG-Bench Hard
    # "leaderboard_gpqa",         # graduate-level QA
    # "leaderboard_mmlu_pro",     # MMLU Professional
    # "leaderboard_musr",         # multi-step reasoning
    # "leaderboard_math_hard",    # hard math problems

    # -- Safety & Fairness --
    # "truthfulqa_mc1",           # truthfulness
    # "toxigen",                  # toxicity detection
    # "winogender",              # gender bias
    # "crows_pairs_english",     # stereotypical bias
    # "bbq",                     # bias benchmark
    # "ethics_cm",               # ethics commonsense

    # -- Toxicity & Ethical Principles --
    # "bigbench_hhh_alignment_multiple_choice",  # HHH alignment
]

# ── Job name ───────────────────────────────────────────────
model_short = EVAL_MODEL.split("/")[-1]
bench_short = EVAL_SUITE or "-".join([b.replace("leaderboard_", "") for b in EVAL_BENCHMARKS])
EVAL_NAME = f"{model_short}-{bench_short}"

# ── Build benchmark list ───────────────────────────────────
if EVAL_SUITE:
    suite = COLLECTIONS[EVAL_SUITE]
    benchmarks_payload = [
        {"id": b["id"], "provider_id": b["provider_id"]}
        for b in suite["benchmarks"]
    ]
    print(f"Suite: {suite['name']}")
else:
    benchmarks_payload = [
        {"id": b, "provider_id": "lm_evaluation_harness"}
        for b in EVAL_BENCHMARKS
    ]

print(f"\n📝 Evaluation: {EVAL_NAME}")
print(f"   Model:      {EVAL_MODEL}")
print(f"   Benchmarks: {[b['id'] for b in benchmarks_payload]}")
print(f"   Secret:     {SECRET_NAME}")
print(f"\n→ Run the next cell to submit.")

## Cell 6 — Submit Evaluation Job

In [ ]:
result = create_job(
    name=EVAL_NAME,
    model_name=EVAL_MODEL,
    model_url=f"{MAAS_BASE}/v1",
    secret_ref=SECRET_NAME,
    benchmarks=benchmarks_payload,
)

job_data = result["data"]
JOB_ID = job_data["resource"]["id"]
state  = job_data["status"]["state"]
msg    = job_data["status"]["message"].get("message", "")

print(f"✅ Job submitted!")
print(f"   ID:      {JOB_ID}")
print(f"   Name:    {EVAL_NAME}")
print(f"   State:   {state}")
print(f"   Message: {msg}")
print(f"\n🔗 Dashboard: {RHOAI_DASHBOARD}/evaluation/{NAMESPACE}")

## Cell 7 — Monitor Job Progress

Polls the EvalHub API until the job completes. You can also watch the Evaluations page in the dashboard.

In [ ]:
completed_job = wait_for_job(JOB_ID, poll_interval=15, timeout=900)

if completed_job:
    state = completed_job["status"]["state"]
    if state in ("complete", "completed"):
        print(f"\n🟢 Evaluation complete!")
    else:
        print(f"\n🔴 Evaluation ended with state: {state}")
        msg = completed_job["status"].get("message", {})
        print(f"   Message: {json.dumps(msg, indent=2)}")

## Cell 8 — View All Evaluation Results

In [ ]:
jobs = list_jobs()

if not jobs:
    print("No evaluation jobs found.")
else:
    print(f"{'Name':<35s} {'State':<12s} {'Model':<25s} {'Benchmarks'}")
    print("─" * 100)
    for j in jobs:
        name  = j.get("name", "?")
        state = j["status"]["state"]
        model = j.get("model", {}).get("name", "?").split("/")[-1]
        benchmarks = [b["id"] for b in j.get("benchmarks", [])]
        print(f"{name:<35s} {state:<12s} {model:<25s} {benchmarks}")
        
        # Show results if available
        results = j.get("results", {})
        evals = results.get("evaluations", [])
        if evals:
            for ev in evals:
                bench_id = ev.get("benchmark_id", "?")
                scores = ev.get("scores", {})
                ev_state = ev.get("state", "?")
                print(f"  └─ {bench_id}: state={ev_state}")
                for metric, value in scores.items():
                    if isinstance(value, float):
                        print(f"       {metric}: {value:.4f}")
        print()

## Cell 9 — Compare Models (run two evaluations)

Submit the same benchmark against two different models and compare.

In [ ]:
# ── Compare two models on the same benchmark ──────────────
COMPARE_MODELS = [
    "publishers/prelude-maas/models/gemma4",
    "publishers/prelude-maas/models/llama-32-3b",
]
COMPARE_BENCHMARKS = [
    {"id": "leaderboard_ifeval", "provider_id": "lm_evaluation_harness"},
]

compare_jobs = []
for model in COMPARE_MODELS:
    short = model.split("/")[-1]
    name = f"compare-{short}-ifeval"
    print(f"Submitting: {name}")
    result = create_job(
        name=name,
        model_name=model,
        model_url=f"{MAAS_BASE}/v1",
        secret_ref=SECRET_NAME,
        benchmarks=COMPARE_BENCHMARKS,
    )
    job_id = result["data"]["resource"]["id"]
    compare_jobs.append({"name": name, "model": short, "job_id": job_id})
    print(f"  ✅ {name} → {job_id}")

print(f"\n🔗 Watch in dashboard: {RHOAI_DASHBOARD}/evaluation/{NAMESPACE}")
print(f"\nBoth jobs submitted! Run Cell 8 to check results once complete.")

## Cell 10 — Cleanup (optional)

Delete evaluation jobs when done.

In [ ]:
# List all jobs first
jobs = list_jobs()
for j in jobs:
    print(f"  {j['resource']['id']}: {j.get('name', '?')} ({j['status']['state']})")

# Uncomment below to delete a specific job:
# evalhub_delete(f"evaluations/jobs/{JOB_ID}")
# print(f"Deleted {JOB_ID}")

# Uncomment below to delete ALL jobs:
# for j in jobs:
#     jid = j["resource"]["id"]
#     evalhub_delete(f"evaluations/jobs/{jid}")
#     print(f"Deleted {jid}: {j.get('name', '?')}")

---

## Reference: EvalHub REST API

| Method | Endpoint | Description |
|--------|----------|-------------|
| `GET`  | `/eval-hub/api/v1/evalhub/health?namespace=NS` | Health check |
| `GET`  | `/eval-hub/api/v1/evaluations/collections?namespace=NS` | List benchmark suites |
| `GET`  | `/eval-hub/api/v1/evaluations/jobs?namespace=NS` | List evaluation jobs |
| `POST` | `/eval-hub/api/v1/evaluations/jobs?namespace=NS` | Create evaluation job |
| `DELETE`| `/eval-hub/api/v1/evaluations/jobs/{id}?namespace=NS` | Delete evaluation job |

### Secret format
```bash
oc create secret generic eval-hub-api-key \\
  --from-literal=api-key=sk-oai-... \\
  -n $NAMESPACE
```

### Job payload
```json
{
  "name": "my-eval",
  "model": {
    "name": "publishers/prelude-maas/models/gemma4",
    "url": "https://maas.../v1",
    "auth": {"secret_ref": "eval-hub-api-key"}
  },
  "benchmarks": [
    {"id": "leaderboard_ifeval", "provider_id": "lm_evaluation_harness"}
  ]
}
```